# OneVoice V2 — fine-tune EnViT5 EN→VI
Notebook này tạo checkpoint **riêng** cho English → Vietnamese. Nó không ghi đè checkpoint VI→EN đã freeze và chỉ dùng `train.csv`/`dev.csv`; `test.csv`, `minimal_pairs.csv` và safety suite chỉ dùng khi đánh giá sau training.

Toàn bộ checkpoint, optimizer và training history được lưu trên Google Drive. Nếu mất GPU hoặc đổi tài khoản Colab, mount lại cùng Drive rồi chạy lại từ cell đầu: training sẽ resume từ epoch hoàn chỉnh gần nhất.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
CACHE_ROOT = DRIVE_ROOT / 'model_cache'
CHECKPOINT_ROOT = DRIVE_ROOT / 'models/envit5_finetuned_en2vi_v1'
for path in (CACHE_ROOT, CHECKPOINT_ROOT):
    path.mkdir(parents=True, exist_ok=True)
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface/hub')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['PYTHONUNBUFFERED'] = '1'

# EnViT5 needs this tested Transformers 4.x/SentencePiece stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'PyYAML', 'torch', 'sacremoses', 'tqdm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
versions = subprocess.run([sys.executable, '-c', "import transformers, tokenizers, sentencepiece; print(transformers.__version__, tokenizers.__version__, sentencepiece.__version__); assert transformers.__version__ == '4.57.1'"], text=True, capture_output=True, check=True)
print('MT versions:', versions.stdout, end='')
print('Persistent EN→VI checkpoint:', CHECKPOINT_ROOT)


In [ ]:
TOTAL_EPOCHS = 3  # Increase only to continue a saved EN→VI run to a later epoch.
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
TRAIN = REPO / 'data/onevoice_construction_v2/train.csv'
DEV = REPO / 'data/onevoice_construction_v2/dev.csv'
STATE = CHECKPOINT_ROOT / 'training_state.pt'
if STATE.is_file():
    print('Resume available:', STATE)
else:
    print('New EN→VI run. Source model: VietAI/envit5-translation')

command = [
    sys.executable, 'scripts/finetune_envit5.py',
    '--model', 'VietAI/envit5-translation',
    '--direction', 'en2vi',
    '--train', str(TRAIN), '--dev', str(DEV),
    '--output', str(CHECKPOINT_ROOT),
    '--epochs', str(TOTAL_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--resume', 'auto',
]
print('>', ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('Fine-tune stopped. Re-run this cell: completed epochs remain on Drive and resume automatically.')


In [ ]:
import json
for name in ('training_history.json', 'run_manifest.json'):
    path = CHECKPOINT_ROOT / name
    if path.is_file():
        display(json.loads(path.read_text(encoding='utf-8')))
best = CHECKPOINT_ROOT / 'best'
if not (best / 'config.json').is_file():
    raise FileNotFoundError('No complete best checkpoint yet; wait for an epoch to finish.')
print('Checkpoint ready for independent EN→VI benchmark:', best)
print('Next: open colab_mt_v2.ipynb and run all; it detects this candidate automatically.')
